# 22 - LSTM with three seeds

How much do the LSTM results depend on the random seed? The reported configuration of notebook 16 (20,000 training sequences), run with three seeds on HDFS and BGL.

**Seeds: 42, 7, 123**, the same as the Isolation Forest and Random Forest studies (notebooks 20 and 21). 42 is the seed of the reported result.

In [1]:
import sys
from pathlib import Path

import pandas as pd
import torch

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.evaluation.cross_system import chosen_rows
from src.models.lstm import DeepLogLSTM, sweep_k
from src.models.unsupervised import time_split_val

DOCS, PROCESSED, MODELS = ROOT / "docs", ROOT / "data" / "processed", ROOT / "models"
H, KS, SEEDS = 10, [1, 2, 3, 5, 9, 12], [42, 7, 123]  # k values as in notebook 16
LABELLED = {"HDFS": ("hdfs", "order"), "BGL": ("bgl", "window")}
METRICS = ["precision", "recall", "f1", "false_alarm_rate"]

## Training

Trained with `scripts/train_lstm_seeds.py` (CPU, in the background): 20,000 evenly spaced normal sequences from the first 60%, embedding 32, 2 layers of 64, h = 10, Adam lr 0.001, batch 512, up to 30 epochs, early stopping after 3 epochs without improvement on a 10% hold-out. The seed sets random, numpy and torch before each run, and so controls the weight initialisation, the batch order and the hold-out.

In [2]:
# weights_only=False: the files also hold the training history (a DataFrame); they are our own files
saved = {(name, seed): torch.load(MODELS / f"{prefix}_lstm_seed{seed}.pt", weights_only=False)
         for name, (prefix, _) in LABELLED.items() for seed in SEEDS}
pd.DataFrame([{"dataset": name, "seed": seed, "train sequences": s["train_sequences"], "epochs": s["epochs"],
               "best epoch (weights kept)": int(s["history"]["val_loss"].idxmin()) + 1,
               "training time (min)": s["train_seconds"] / 60} for (name, seed), s in saved.items()]) \
    .style.format({"train sequences": "{:,}", "training time (min)": "{:.1f}"}).hide(axis="index")

dataset,seed,train sequences,epochs,best epoch (weights kept),training time (min)
HDFS,42,"20,000",19,16,6.6
HDFS,7,"20,000",27,24,9.9
HDFS,123,"20,000",17,14,6.5
BGL,42,"20,000",15,12,6.9
BGL,7,"20,000",19,16,7.0
BGL,123,"20,000",17,14,5.7


## Choose k on validation, test once

Same rule as before: a sequence is flagged if an observed event is outside the top k predictions, or if it has an event never seen in the training sequences. k is chosen by validation F1 for each seed.

In [3]:
rows, val_rows = [], []
for name, (prefix, order_col) in LABELLED.items():
    _, val, test = time_split_val(pd.read_parquet(PROCESSED / f"{prefix}_sequences.parquet"), order_col=order_col)
    for seed in SEEDS:
        s = saved[(name, seed)]
        model = DeepLogLSTM(num_events=len(s["vocab"]), **{k: v for k, v in s["config"].items() if k != "h"})
        model.load_state_dict(s["state_dict"])
        model.eval()
        known = set(s["train_events"])
        v, *_ = sweep_k(model, val["EventSequence"].tolist(), s["vocab"], known, KS, val["Label"].to_numpy(), H)
        t, *_ = sweep_k(model, test["EventSequence"].tolist(), s["vocab"], known, KS, test["Label"].to_numpy(), H)
        v, t = v.assign(dataset=name, model="LSTM"), t.assign(dataset=name, model="LSTM")
        val_rows.append(v.assign(seed=seed))
        best = chosen_rows(v, t, "k").iloc[0]  # best validation F1, first k on ties (as in notebook 16)
        rows.append({"dataset": name, "seed": seed, "chosen_k": int(best["k"]), "epochs": s["epochs"],
                     **{m: best[m] for m in METRICS}})

results = pd.DataFrame(rows)
results.round(4).to_csv(DOCS / "seed_results_lstm.csv", index=False)
results.style.format({m: "{:.4f}" for m in METRICS}).hide(axis="index")

dataset,seed,chosen_k,epochs,precision,recall,f1,false_alarm_rate
HDFS,42,5,19,0.8017,0.4300,0.5598,0.0023
HDFS,7,5,27,0.8641,0.4491,0.5910,0.0016
HDFS,123,5,17,0.7883,0.4171,0.5456,0.0025
BGL,42,12,15,0.0974,0.9925,0.1774,0.7410
BGL,7,12,19,0.0980,0.9925,0.1784,0.7363
BGL,123,12,17,0.0978,0.9925,0.1781,0.7375


Validation F1 by k, per seed:

In [4]:
pd.concat(val_rows).pivot_table(index=["dataset", "seed"], columns="k", values="f1", sort=False).round(3)

k                1      2      3      5      9      12
dataset seed                                          
HDFS    42    0.090  0.321  0.381  0.532  0.463  0.430
        7     0.090  0.326  0.375  0.604  0.414  0.398
        123   0.090  0.326  0.438  0.491  0.421  0.406
BGL     42    0.241  0.256  0.268  0.276  0.286  0.289
        7     0.241  0.256  0.268  0.278  0.288  0.292
        123   0.241  0.256  0.267  0.278  0.288  0.291

## Mean and standard deviation over the seeds

Sample standard deviation (n - 1). The reported values are the notebook 16 results (seed 42).

In [5]:
summary = results.groupby("dataset", sort=False)[METRICS].agg(["mean", "std"])
summary.columns = [f"{m}_{s}" for m, s in summary.columns]
summary = summary.reset_index()
reported = pd.read_csv(DOCS / "cross_system_results.csv").query("model == 'LSTM' and dataset == `chosen on`").set_index("dataset")
summary["reported_f1"] = summary["dataset"].map(reported["f1"])
summary["seed_42_f1"] = summary["dataset"].map(results[results["seed"] == 42].set_index("dataset")["f1"])
summary.style.format({c: "{:.4f}" for c in summary.columns if c != "dataset"}).hide(axis="index")

dataset,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,false_alarm_rate_mean,false_alarm_rate_std,reported_f1,seed_42_f1
HDFS,0.8180,0.0404,0.4321,0.0161,0.5655,0.0232,0.0021,0.0005,0.5598,0.5598
BGL,0.0978,0.0003,0.9925,0.0000,0.1780,0.0005,0.7383,0.0024,0.1774,0.1774


In [6]:
pd.DataFrame({m: summary.set_index("dataset").apply(lambda r: f"{r[m + '_mean']:.3f} ± {r[m + '_std']:.3f}", axis=1)
              for m in METRICS})

,precision,recall,f1,false_alarm_rate
dataset,,,,
HDFS,0.818 ± 0.040,0.432 ± 0.016,0.565 ± 0.023,0.002 ± 0.000
BGL,0.098 ± 0.000,0.992 ± 0.000,0.178 ± 0.000,0.738 ± 0.002


## Findings

- **Seed 42 reproduces notebook 16 exactly** (HDFS F1 0.5598, BGL F1 0.1774, same epochs and k), so training on CPU is deterministic here.
- **BGL is very stable:** F1 0.178 ± 0.001, k = 12 for every seed.
- **HDFS varies more:** F1 0.566 ± 0.023 (0.546 to 0.591), mostly through precision (0.79 to 0.86). k = 5 for every seed. The number of epochs also differs (17 to 27), so the seed changes where training stops.
- The LSTM is the least stable of the three models on HDFS (Isolation Forest ± 0.007, Random Forest ± 0.0002), but the conclusions hold for every seed: it stays well below PCA (0.838) and Random Forest (0.999) on HDFS.